# Inferencia local con Ollama

**LLMs open source corriendo en tu máquina.** En esta lección usamos [Ollama](https://ollama.com)
como runtime local y lo consumimos desde **LangChain** (`langchain-ollama`), con la misma interfaz
con la que consumiríamos OpenAI o Anthropic — pero sin API keys, sin costo por token y sin que
los datos salgan de tu computador.

Antes de ejecutar:

1. Instala Ollama y déjalo corriendo (la app de escritorio, o `ollama serve` en una terminal).
2. Descarga el modelo de la lección: `ollama pull llama3.2` (~2.0 GB, una sola vez).

Más detalle (qué es Ollama, instalación por sistema operativo) en el [README](README.md) de la lección.


In [ ]:
# Esta lección corre LOCAL (el modelo vive en tu máquina) — usa el entorno uv del README.
# Si la corres en otro entorno, descomenta: %pip install -q langchain-ollama==1.1.0 langchain-core==1.4.9
import json
import urllib.request

# Cambia MODEL por cualquier modelo que tengas descargado (`ollama list`).
MODEL = "llama3.2"
OLLAMA_URL = "http://localhost:11434"

try:
    with urllib.request.urlopen(f"{OLLAMA_URL}/api/version", timeout=3) as resp:
        version = json.load(resp)["version"]
    print(f"Servidor Ollama corriendo (v{version}) en {OLLAMA_URL}")
except Exception as exc:
    raise RuntimeError(
        "No hay un servidor Ollama en localhost:11434 — abre la app de Ollama "
        "o corre `ollama serve` en una terminal."
    ) from exc

with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=3) as resp:
    disponibles = [m["name"] for m in json.load(resp)["models"]]
print("Modelos locales:", ", ".join(disponibles) or "(ninguno)")
if not any(n == MODEL or n.startswith(f"{MODEL}:") for n in disponibles):
    print(f"⚠️  '{MODEL}' no está descargado — corre `ollama pull {MODEL}` primero.")


## Invocación con LangChain

`ChatOllama` implementa la misma interfaz que cualquier chat model de LangChain (`invoke`,
`stream`, mensajes `system`/`human`), así que todo lo que ya sabes de cadenas y agentes aplica
igual con un modelo local. Usamos la misma traducción ES→EN que verás en la lección 3 (Groq),
para poder comparar después velocidad local vs. nube.


In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model=MODEL, temperature=0)

messages = [
    ("system", "Eres un traductor. Traduce la frase del usuario del español al inglés."),
    ("human", "Me encanta programar."),
]

respuesta = llm.invoke(messages)
print(respuesta.content)


## Streaming

Con `.stream()` recibimos la respuesta token a token — la experiencia de "va escribiendo".
En un modelo local esto además te deja *ver* la velocidad real de generación de tu hardware.


In [ ]:
pregunta = [("human", "Explica en dos frases qué es un modelo open-weights.")]

for chunk in llm.stream(pregunta):
    print(chunk.content, end="", flush=True)
print()


## Velocidad local (tokens/segundo)

Ollama reporta en `response_metadata` cuántos tokens generó (`eval_count`) y cuánto demoró
(`eval_duration`, en **nanosegundos**). Con eso calculamos los t/s reales de tu máquina.
Guarda este número: en la lección 3 medimos lo mismo contra Groq (~500–900 t/s con los `gpt-oss`).


In [ ]:
respuesta = llm.invoke(messages)

meta = respuesta.response_metadata
tokens = meta["eval_count"]              # tokens generados
segundos = meta["eval_duration"] / 1e9   # eval_duration viene en nanosegundos

print(f"Modelo: {MODEL}")
print(f"Tokens generados: {tokens}")
print(f"Velocidad local: {tokens / segundos:.1f} t/s")


## Cierre

El mismo modelo abierto puede vivir donde te convenga: **en tu máquina** (privacidad, costo
cero, velocidad limitada por tu hardware) o **en la nube especializada** como Groq
(lección 3, cientos de t/s). Y como los pesos son abiertos, también puedes **adaptarlos con
fine-tuning** — eso es la lección 2.
